# End-to-End LLM Evaluation on OpenShift: Benchmarks, MLflow, and OCI Exports

This notebook walks through running an LLM evaluation on an OpenShift cluster using the **`eval-hub-sdk[client]`** Python SDK. It covers tracking experiment results in MLflow and exporting evaluation artifacts to an OCI-compatible registry. By the end, you will have:

1. Connected to an EvalHub instance using the **`eval-hub-sdk[client]`** Python client
2. Submitted an evaluation job using the **lm_evaluation_harness** provider with the **arc_easy** benchmark
3. Logged experiment results to **MLflow**
4. Exported evaluation artifacts to an **OCI-compatible registry**

## Prerequisites

### Cluster Requirements

- An OpenShift cluster with **EvalHub deployed**. See the [deployment guide](https://eval-hub.github.io/deployment/openshift-setup/) for setup instructions.
- An LLM model endpoint accessible from the cluster, with its API key and CA certificate stored as a Kubernetes secret in the tenant namespace. See [model authentication](https://eval-hub.github.io/guides/model-authentication/) for details.
- **MLflow** installed and configured with EvalHub. This is required for experiment tracking.
- An **OCI-compatible registry** (e.g., Quay.io) for exporting result artifacts. See the [quickstart guide](https://eval-hub.github.io/getting-started/quickstart/#export-results-to-oci-registry).

### Notebook Environment

Install the required Python packages:
```bash
pip install "eval-hub-sdk[client]" "oras"
```

## Gather Connection Details

The values below are specific to your cluster. For example, assuming EvalHub is deployed in namespace `prabhu` and there is a service account `dataplane-sa` in the `dataplane` namespace:

```bash
export EVALHUB_TENANT="dataplane"
export TENANT_SA="dataplane-sa"
export EVALHUB_SERVICE_NS="prabhu"
```

Generate a short-lived service account token:
```bash
oc create token "${TENANT_SA}" -n "${EVALHUB_TENANT}" --duration=1h
```

Get the EvalHub route URL:
```bash
echo "https://$(oc get route evalhub -n "${EVALHUB_SERVICE_NS}" -o jsonpath='{.spec.host}')"
```

See the [multi-tenancy overview](https://eval-hub.github.io/architecture/multi-tenancy/#overview) for more details on tenants and service accounts.

## Configure Connection

Fill in the values below with the connection details gathered above.

In [ ]:
# EvalHub service URL (from `oc get route`)
# e.g.
# evalhub_url = "https://evalhub-prabhu.apps.rosa.prabhu-comhub.xqmp.p3.openshiftapps.com"
evalhub_url = "..."

# Tenant namespace
# e.g.
# evalhub_tenant = "dataplane"
evalhub_tenant = "..."

# Service account token (from `oc create token`)
evalhub_token = "..."

## Import the EvalHub SDK

In [27]:
from evalhub import (
    SyncEvalHubClient,
    JobSubmissionRequest,
    BenchmarkConfig,
    ModelConfig,
    JobStatus,
    ExperimentConfig,
    ExperimentTag,
)
from evalhub.models.api import ModelAuth

## Connect to EvalHub

Create a client and verify the service is reachable.

In [28]:
client = SyncEvalHubClient(
    base_url=evalhub_url,
    auth_token=evalhub_token,
    insecure=True,
    tenant=evalhub_tenant,
)
try:
    # Check health
    health = client.health()
    print(f"✓ EvalHub is healthy: {health['status']}")
    print(f"  Version: {health.get('version', 'unknown')}")
    print(f"  Uptime: {health.get('uptime_seconds', 0):.1f}s")
except Exception as e:
    print(f"✗ Failed to connect: {e}")

TLS verification disabled - skipping CA bundle detection
TLS verification disabled (insecure mode)


✓ EvalHub is healthy: healthy
  Version: unknown
  Uptime: 0.0s


## List Available Providers

In [29]:
providers = client.providers.list()
print(f"✓ Found {len(providers)} providers:")
for provider in providers:
    print(f"  - {provider.resource.id}: {provider.name}")

✓ Found 8 providers:
  - lm_evaluation_harness: LM Evaluation Harness
  - lighteval: Lighteval
  - guidellm: GuideLLM
  - garak: Garak
  - afc721e4-2cd9-4c93-8d9f-4379682f5792: SWE-bench
  - aebfd3ff-5d83-455b-afe6-6f5794f3876e: SWE-bench
  - 6c6c3340-c87a-4e0c-bb05-7256a536b207: SWE-bench
  - 370be638-8aa9-4e72-b384-feccb81579e8: SWE-bench


## List Benchmarks for `lm_evaluation_harness`

Query the available benchmarks under the provider we will use.

In [30]:
provider_id = "lm_evaluation_harness"
benchmarks = client.benchmarks.list(provider_id=provider_id)
print(f"\n✓ Found {len(benchmarks)} benchmarks for {provider_id}:")
for benchmark in benchmarks:
    print(f"  - {benchmark.id}: {benchmark.name}")


✓ Found 188 benchmarks for lm_evaluation_harness:
  - arc_easy: Basic science Q&A
  - AraDiCE_boolq_lev: Yes/no question answering (Levantine Arabic)
  - blimp: Grammar & syntax judgment
  - blimp_anaphor_gender_agreement: Pronoun gender matching
  - blimp_animate_subject_trans: Animacy constraints on verbs
  - blimp_coordinate_structure_constraint_complex_left_branch: Complex sentence structure
  - blimp_determiner_noun_agreement_2: Determiner-noun number agreement
  - blimp_determiner_noun_agreement_with_adj_2: Agreement across adjectives (variant 2)
  - blimp_determiner_noun_agreement_with_adjective_1: Agreement across adjectives (variant 1)
  - blimp_existential_there_object_raising: Existential construction (object)
  - blimp_existential_there_subject_raising: Existential construction (subject)
  - blimp_intransitive: Verb argument structure
  - blimp_irregular_plural_subject_verb_agreement_1: Irregular plural agreement
  - blimp_left_branch_island_simple_question: Question forma

## Configure the Model

Provide the URL, name, and Kubernetes secret for the model you want to evaluate. The secret must exist in the tenant namespace and contain the API key (and optionally a CA certificate). See the [model authentication guide](https://eval-hub.github.io/guides/model-authentication/#scenario-1-api-key-and-ca-certificate) for how to create one.

In [ ]:
# e.g.
# model_url = "https://vllm-llama-prabhu.apps.rosa.prabhu-comhub.xqmp.p3.openshiftapps.com/v1"
# model_secret = "vllm-llama-api-key"
# model_name = "meta-llama/Llama-3.2-1B-Instruct"

model_url = "..."
model_secret = "..."
model_name = "..."

## Submit an Evaluation Job

Configure and submit a job that runs the `arc_easy` benchmark using `lm_evaluation_harness`, logs results to MLflow, and exports artifacts to an OCI registry.

In [32]:
# this is the benchmark id, you can find it from the list of benchmarks above
benchmark_id = "arc_easy"

The `lm_evaluation_harness` framework requires a tokenizer. Here we use `google/flan-t5-small` — replace it with a tokenizer appropriate for your model if needed.

In [33]:
tokenizer_model = "google/flan-t5-small"

In [ ]:
# The job name of your choice
job_name = "my-ev-job1"

# The MLFlow experiment name of your choice
mlflow_experiment_name = "my-exp1"

### Configure OCI Export

To export result artifacts to an OCI registry, create a `docker-registry` secret with write access to the target repository and make it available in the tenant namespace. See the [OCI export documentation](https://eval-hub.github.io/getting-started/quickstart/#export-results-to-oci-registry) for details.

```bash
oc create secret docker-registry my-oci-creds \
    --docker-server=quay.io \
    --docker-username=<username> \
    --docker-password=<password> \
    --namespace <tenant-namespace>
```

In [ ]:
oci_registry_url = "quay.io"

# e.g.
# oci_registry_repo = "gnaulak/testp1/eval-results"
# oci_registry_secret = "my-oci-creds-b"

oci_registry_repo = "..."
oci_registry_secret = "..."

In [ ]:
from evalhub import EvaluationExports, EvaluationExportsOCI, OCIConnectionConfig, OCICoordinates

job_request = JobSubmissionRequest(
    name=job_name,
    model=ModelConfig(
        url=model_url, name=model_name, auth=ModelAuth(secret_ref=model_secret)
    ),
    benchmarks=[
        BenchmarkConfig(
            id=benchmark_id,
            provider_id=provider_id,
            parameters={
                "num_examples": 5,
                "tokenizer": tokenizer_model,
            },
        )
    ],
    experiment=ExperimentConfig(
        name=mlflow_experiment_name,
        tags=[
            ExperimentTag(key="env", value="dev"),
        ],
    ),
    exports=EvaluationExports(
        oci=EvaluationExportsOCI(
            coordinates=OCICoordinates(
                oci_host=oci_registry_url,
                oci_repository=oci_registry_repo,
            ),
            k8s=OCIConnectionConfig(
                connection=oci_registry_secret,
            ),
        )
    ),
)

job = client.jobs.submit(job_request)
submitted_job_id = job.id
print(f"✓ Job submitted successfully")
print(f"  Job ID: {submitted_job_id}")
print(f"  State: {job.state}")
print(f"  Created: {job.resource.created_at}")

In [44]:
# Check status
updated_job = client.jobs.get(submitted_job_id)
print(f"\n✓ Current job state: {updated_job.state}")
if updated_job.status and updated_job.status.message:
    print(f"  Message: {updated_job.status.message.message}")


✓ Current job state: JobStatus.PENDING
  Message: Evaluation job created


## Wait for the Job to Complete

Poll the job status until it reaches `COMPLETED` or `FAILED`.

In [45]:
import time

while updated_job.state not in (JobStatus.COMPLETED, JobStatus.FAILED):
    print(f"\n✓ Current job state: {updated_job.state}")
    time.sleep(5)
    updated_job = client.jobs.get(submitted_job_id)

if updated_job.state == JobStatus.FAILED:
    if updated_job.status and updated_job.status.message:
        print(f"Job failed: {updated_job.status.message.message}")
    else:
        print("Job failed: Unknown")
else:
    print(f"✓ Job completed successfully")


✓ Current job state: JobStatus.PENDING

✓ Current job state: JobStatus.PENDING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING
✓ Job completed successfully


## Inspect the Completed Job

Dump the full job object to see all fields including status, results, experiment metadata, and export details.

In [ ]:
print(updated_job.model_dump_json(indent=2))

{
  "resource": {
    "id": "5db449a4-5bc0-4ba7-8106-aaa20e3019f7",
    "tenant": "dataplane",
    "created_at": "2026-06-16T05:19:47.705471Z",
    "updated_at": "2026-06-16T05:20:09.516827Z",
    "mlflow_experiment_id": "2",
    "message": null
  },
  "status": {
    "state": "completed",
    "message": {
      "message": "Evaluation job is completed",
      "message_code": "evaluation_job_updated"
    },
    "benchmarks": [
      {
        "id": "arc_easy",
        "provider_id": "lm_evaluation_harness",
        "benchmark_index": 0,
        "status": "completed",
        "error_message": null,
        "started_at": null,
        "completed_at": "2026-06-16T05:20:07.643559Z"
      }
    ]
  },
  "results": {
    "benchmarks": [
      {
        "id": "arc_easy",
        "provider_id": "lm_evaluation_harness",
        "benchmark_index": 0,
        "metrics": {
          "acc": 0.2,
          "acc_norm": 0.2,
          "acc_norm_stderr": 0.20000000000000004,
          "acc_stderr": 0.20

### Benchmark Metrics

Extract the metrics returned by the `arc_easy` benchmark.

In [ ]:
print("Benchmark metrics:", updated_job.results.benchmarks[0].metrics)

### MLflow Run ID

The evaluation results are also logged to MLflow. Use the run ID below to find the run in the MLflow UI.

In [59]:
updated_job.results.benchmarks[0].mlflow_run_id

'e8f8926ec4de44bcaa5d7c9141cb9e6b'

The MLflow experiment `my-exp1` (the name we provided) with this run ID will be visible in the MLflow UI under the tenant's workspace.

## Verify OCI Registry Export

The evaluation artifacts were exported to the OCI registry. Let's verify by fetching the OCI manifest and inspecting the contents.

In [74]:
oci_reference = updated_job.results.benchmarks[0].artifacts["oci_reference"]
oci_reference

'quay.io/gnaulak/testp1/eval-results:evalhub-9ffbfa2a10f1a273deb1df678f1ac70cda1962ca89b62d049de3428cfeb4948a@sha256:e50fdf9d0b643efb8325e077c92cf6ab4fb0a14625718a120975d86390024e98'

In [ ]:
# OCI registry credentials for verification (used by the oras client below)
# e.g.
# oci_username = "<oci-username>"
# oci_password = "<oci-password>"
oci_username = "..."
oci_password = "..."

In [68]:
import oras.client
import json
import tempfile
import os

oras_client = oras.client.OrasClient()
oras_client.login(
    hostname="quay.io",
    username=oci_username,
    password=oci_password,
)


# get manifest from oci_reference
manifest = oras_client.get_manifest(oci_reference)
print(f"\nManifest:\n{json.dumps(manifest, indent=2)}")



Manifest:
{
  "schemaVersion": 2,
  "mediaType": "application/vnd.oci.image.manifest.v1+json",
  "artifactType": "application/vnd.eval-hub.github.io",
  "config": {
    "mediaType": "application/vnd.oci.image.config.v1+json",
    "size": 299,
    "digest": "sha256:978ec14d5e685db5e2035705ad1589f1b40f8a72fd9d218fe0d363a1f445db1b"
  },
  "layers": [
    {
      "mediaType": "application/vnd.oci.image.layer.v1.tar+gzip",
      "size": 1146,
      "digest": "sha256:40bb1706f33b8b559a430fcb38e468c8960ffd3d7f0fc2f0427318b9a95293f5",
      "annotations": {
        "olot.layer.content.name": "results_5db449a4-5bc0-4ba7-8106-aaa20e3019f7.json"
      }
    }
  ],
  "annotations": {
    "io.github.eval-hub.job_id": "5db449a4-5bc0-4ba7-8106-aaa20e3019f7",
    "io.github.eval-hub.benchmark_id": "arc_easy",
    "io.github.eval-hub.provider_id": "lm_evaluation_harness",
    "org.opencontainers.image.created": "2026-06-16T05:20:07.650508+00:00",
    "io.github.eval-hub.benchmark": "arc_easy",
    "io

### Pull and Inspect Artifact Contents

Pull the OCI artifact and extract the evaluation results JSON from the tarball.

In [ ]:
import tarfile

with tempfile.TemporaryDirectory() as tmpdir:
    pulled = oras_client.pull(target=oci_reference, outdir=tmpdir)
    print(f"Pulled {len(pulled)} files:")

    for filepath in pulled:
        print(f"\n--- {os.path.basename(filepath)} ---")
        actual_path = os.path.join(tmpdir, os.path.basename(filepath))
        if not os.path.exists(actual_path):
            for f in os.listdir(tmpdir):
                actual_path = os.path.join(tmpdir, f)
                break

        if tarfile.is_tarfile(actual_path):
            with tarfile.open(actual_path, "r:gz") as tar:
                for member in tar.getmembers():
                    f = tar.extractfile(member)
                    if f:
                        content = f.read().decode("utf-8")
                        print(f"\n  [{member.name}]")
                        try:
                            parsed = json.loads(content)
                            print(json.dumps(parsed, indent=2))
                        except json.JSONDecodeError:
                            print(content)
        else:
            with open(actual_path, "r") as f:
                content = f.read()
                try:
                    parsed = json.loads(content)
                    print(json.dumps(parsed, indent=2))
                except json.JSONDecodeError:
                    print(content)

Pulled 1 files:

--- sha256:40bb1706f33b8b559a430fcb38e468c8960ffd3d7f0fc2f0427318b9a95293f5 ---
{
  "id": "5db449a4-5bc0-4ba7-8106-aaa20e3019f7",
  "benchmark_id": "arc_easy",
  "benchmark_index": 0,
  "model_name": "meta-llama/Llama-3.2-1B-Instruct",
  "results": [
    {
      "metric_name": "acc",
      "metric_value": 0.2,
      "metric_type": "float",
      "confidence_interval": null,
      "num_samples": null,
      "metadata": {}
    },
    {
      "metric_name": "acc_stderr",
      "metric_value": 0.20000000000000004,
      "metric_type": "float",
      "confidence_interval": null,
      "num_samples": null,
      "metadata": {}
    },
    {
      "metric_name": "acc_norm",
      "metric_value": 0.2,
      "metric_type": "float",
      "confidence_interval": null,
      "num_samples": null,
      "metadata": {}
    },
    {
      "metric_name": "acc_norm_stderr",
      "metric_value": 0.20000000000000004,
      "metric_type": "float",
      "confidence_interval": null,
      "

## Clean Up

Delete the evaluation job and close the client connection.

In [ ]:
# run this cell to delete the job with hard_delete=True
client.jobs.cancel(submitted_job_id, hard_delete=True)

In [16]:
## Close the client
client.close()